# Modifying 1000G data to exemplify IDEAL-GENOM usage

In this notebook we will modify the 1000G dataset by adding synthetic phenotype data to demonstrate the capabilities of the IDEAL-GENOM library. The original 1000 Genomes dataset does not include phenotype information, so we will randomly assign case/control status to enable quality control and GWAS analysis examples.

Rather than assuming the 1000 Genomes reference panel has already been downloaded, this notebook fetches it itself using `Fetcher1000Genome` from `ideal_genom.core.get_references` — the same helper used internally by `AncestryQC`. The fetched files are cached under `ideal_genom/data/1000genomes_build_38/`, so re-running this notebook (or any other module that needs the same reference panel) will reuse the cached download instead of re-fetching it.

Firstly, let us import the required libraries.

In [ ]:
import sys
import os

from pathlib import Path

import pandas as pd
import numpy as np


Now, let us add the library path to PATH.

In [ ]:
# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

print(f"Library path added to sys.path: {library_path}")

Then, we import a function from the library to run PLINK2 commands, and the `Fetcher1000Genome` class used to fetch the 1000 Genomes reference dataset.

In [ ]:
from ideal_genom.core.executor import run_plink2
from ideal_genom.core.get_references import Fetcher1000Genome


Let us set up the directory structure for our test data. We'll create paths for:
- `inputData`: Original and processed genetic data files
- `outputData`: Analysis results and quality control outputs
- `config`: Configuration files for the pipeline

In [3]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

test_data = DATA_PATH / 'test_data'
if not test_data.exists():
    test_data.mkdir(parents=True, exist_ok=True)

In [ ]:
inputData = test_data / 'inputData'
outputData = test_data / 'outputData'
config    = test_data / 'config'

In the next step we check if the updated 1000G files with phenotypes already exist (.bim, .fam, .bed files in `inputData`). We then fetch the 1000 Genomes reference panel via `Fetcher1000Genome`: it downloads the PLINK2 pgen/pvar/psam files, converts them to PLINK1.9 binaries, and caches everything under `ideal_genom/data/1000genomes_build_38/`. The fetch is itself idempotent — if the cached binaries already exist, no download or conversion happens — so it is safe to run on every execution of this notebook.

In [ ]:
inputData.mkdir(parents=True, exist_ok=True)
outputData.mkdir(parents=True, exist_ok=True)
config.mkdir(parents=True, exist_ok=True)

updated_pheno = (inputData / '1kG_phase3_GRCh38_updated.bim').exists() and (inputData / '1kG_phase3_GRCh38_updated.fam').exists() and (inputData / '1kG_phase3_GRCh38_updated.bed').exists()

fetcher = Fetcher1000Genome(destination=DATA_PATH / '1000genomes_build_38', build='38')
fetcher.get_1000genomes()
fetcher.get_1000genomes_binaries()

print(f"1000 Genomes binaries ready at: {fetcher.bed_file}")

Since the 1000G dataset does not have phenotypes, we are going to randomly assign binary phenotypes to each sample. We'll use:
- **1 = Control** (unaffected)
- **2 = Case** (affected)

This phenotype information will be used later during variant QC and GWAS examples. We set a random seed (42) to ensure reproducibility of the phenotype assignments. This step (and the ones that follow) is skipped if the updated files already exist in `inputData`.

In [ ]:
if not updated_pheno:
    df_1kg_fam = pd.read_csv(
        fetcher.fam_file,
        sep=r'\s+',
        header=None,
        names=["FID", "IID", "PAT", "MAT", "SEX", "PHENO"],
        engine='python'
    )
    df_1kg_fam.head()
else:
    print("Updated phenotype files already exist in inputData. Skipping phenotype assignment.")

In [ ]:
if not updated_pheno:
    np.random.seed(42)
    df_1kg_fam["PHENO"] = np.random.choice([1, 2], size=len(df_1kg_fam))

Now we save the updated phenotype data to a new .fam file in `inputData`. This file will be used by PLINK2 to update the fetched dataset with our synthetic phenotypes.

In [ ]:
if not updated_pheno:
    df_1kg_fam.to_csv(
        inputData / '1kG_phase3_GRCh38_to_update.fam',
        sep=' ',
        header=False,
        index=False
    )

Next, we use PLINK2 to create a new binary fileset (.bed/.bim/.fam) that incorporates our updated phenotype information. The `--make-bed` command generates the binary format, the `--bfile` flag points directly at the binaries `fetcher` produced (no intermediate copy), and the `--fam` flag specifies our custom .fam file with the new phenotypes.

PLINK2 prints a lot of console text for this; we capture it into `plink2_log` to keep the notebook readable — run `plink2_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture plink2_log
if not updated_pheno:
    plink2_args = [
                "--bfile", str(fetcher.bed_file.with_suffix('')),
                "--make-bed",
                "--out", str(inputData / '1kG_phase3_GRCh38_updated'),
                "--fam", str(inputData / '1kG_phase3_GRCh38_to_update.fam')
            ]

    run_plink2(plink2_args)

In [ ]:
if not updated_pheno:
    print(f"PLINK2 make-bed completed. Updated files written to: {inputData / '1kG_phase3_GRCh38_updated'}")
else:
    print("Updated PLINK binaries already exist in inputData. Skipping PLINK2 step.")

Finally, we clean up the temporary `.fam` file used to feed PLINK2 the new phenotypes. We deliberately leave the fetched 1000 Genomes reference binaries untouched under `ideal_genom/data/1000genomes_build_38/` — they're a shared, cached download that other modules (e.g. `AncestryQC`) reuse, so this notebook never deletes them.

In [ ]:
if not updated_pheno:
    (inputData / '1kG_phase3_GRCh38_to_update.fam').unlink(missing_ok=True)